In [ ]:
# 0 — Mount Google Drive và đặt đường dẫn
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/vi-sub-correction')
CKPT_ROOT  = DRIVE_ROOT / 'checkpoints'
DATA_DRIVE = DRIVE_ROOT / 'data'
CORPUS_DRIVE = DRIVE_ROOT / 'corpus'
for d in (CKPT_ROOT, DATA_DRIVE):
    d.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path('/content/vi-sub-corretion')
DATA_DIR = REPO_DIR / 'data' / 'synthetic'

print('Da ket noi Drive.')
print('Checkpoint ben vung :', CKPT_ROOT)
print('Du lieu tam (/content):', DATA_DIR)

In [ ]:
# 1 — Kiểm tra GPU, lấy code và cài thư viện
!nvidia-smi --query-gpu=name,memory.total --format=csv

if not (REPO_DIR / '.git').exists():
    import subprocess
    from datetime import datetime
    if REPO_DIR.exists():
        backup = REPO_DIR.with_name(REPO_DIR.name + '_backup_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
        REPO_DIR.rename(backup)
        print('Đã giữ lại thư mục cũ tại:', backup)
    subprocess.run(['git', 'clone', 'https://github.com/lhhoang5425/vi-sub-corretion.git', str(REPO_DIR)], check=True)

DATA_DIR.mkdir(parents=True, exist_ok=True)
%cd /content/vi-sub-corretion
!git pull
!pip install -r requirements.txt -q
!pip install -q openpyxl

print('--- 3 commit gan nhat ma Colab dang chay ---')
!git log --oneline -3

In [ ]:
# 2 — KIỂM TRA KẾT NỐI trước khi train
# Mất ~10 giây, nhưng nếu bỏ qua mà có lỗi thì mất 1-2 giờ mới biết.
import importlib.util
import subprocess
from pathlib import Path

import torch

problems = []


def check(label, cond, hint=''):
    print(f'{"OK   " if cond else "LOI  "} {label}')
    if not cond:
        problems.append(label)
        if hint:
            print(f'       -> {hint}')


check('GPU khả dụng', torch.cuda.is_available(),
      'Runtime > Change runtime type > T4 GPU. gg1 có thể đã hết hạn mức -> dùng gg2.')
check('Drive đã mount', Path('/content/drive/MyDrive').exists(), 'Chạy lại cell 0.')
check(f'Thư mục dự án trên Drive ({DRIVE_ROOT.name})', DRIVE_ROOT.exists(),
      'Kiểm tên thư mục trên Drive rồi sửa DRIVE_ROOT ở cell 0.')

zips = [p for p in CKPT_ROOT.glob('*.zip') if 'bmd1905' in p.name.lower()] if CKPT_ROOT.exists() else []
ref_ready = (CKPT_ROOT / 'ref_lan4_legacy' / 'config.json').is_file()
print(f'TÙY CHỌN: mốc Lần 4 — {len(zips)} ZIP, đã giải nén: {ref_ready} (cell 8).')

txts = list(CORPUS_DRIVE.glob('*.txt'))
print(f'TÙY CHỌN: corpus riêng trên Drive — {len(txts)} tệp .txt (cell 6).')

tests = sorted(p.name for p in Path('data/gold_test').rglob('*.jsonl'))
check(f'Tập test trong repo ({len(tests)} tệp)', len(tests) >= 4, 'Chạy git pull ở cell 1.')

need = ['torch', 'transformers', 'datasets', 'sacrebleu', 'rouge_score']
missing = [m for m in need if importlib.util.find_spec(m) is None]
check('Thư viện đã cài', not missing, f'Thiếu {missing} — chạy lại cell 1.')

print('\nCommit đang chạy:',
      subprocess.run(['git', 'log', '-1', '--format=%h %s'],
                     capture_output=True, text=True).stdout.strip())

print()
if problems:
    print(f'=== CÒN {len(problems)} VẤN ĐỀ — SỬA TRƯỚC KHI TRAIN ===')
else:
    print('=== SẴN SÀNG CHẠY TIẾP ===')


In [ ]:
# 3 — Sinh train và validation cho ba hồ sơ nhiễu
import json
import re
from shutil import copyfile
from datasets import load_dataset
from src.data_processing.corpus_builder import build_noisy_corpus, format_stats, write_jsonl

# 100k câu SẠCH (nạp dư vì nguồn trùng lặp ~21%), sinh 3 hồ sơ nhiễu để so sánh.
NUM_TRAIN_CLEAN = 100_000
NUM_FETCH = 220_000
SEED = 42
VARIANTS = ['legacy', 'asr_hard', 'whisper_like']
VAL_RATIO = 0.1

raw = load_dataset('ShynBui/Vietnamese_spelling_error', split=f'train[:{NUM_FETCH}]')
clean = [(r['text'] or '').strip() for r in raw]

# Khử trùng lặp rồi chia theo hash nội dung -> không rò rỉ val<->train (lỗi của Lần 4)
corpus_by_profile = {
    p: build_noisy_corpus(clean, profile=p, val_ratio=VAL_RATIO, seed=SEED, max_clean=NUM_TRAIN_CLEAN)
    for p in VARIANTS
}
print(format_stats(corpus_by_profile[VARIANTS[0]]['stats']))

TONE = re.compile(r'[àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ]', re.I)
gold = [json.loads(l) for l in open('data/gold_test/gd1_basic/sample_test.jsonl', encoding='utf-8') if l.strip()]
gold_lost = sum(1 for g in gold if not TONE.search(g['noisy'])) / len(gold) * 100
print(f'\nĐề thi gốc yêu cầu khôi phục dấu thanh cho {gold_lost:.0f}% câu\n')

print(f'{"HỒ SƠ":<14}{"train":>9}{"val":>8}{"% mất trắng dấu":>18}')
print('-' * 50)
for prof in VARIANTS:
    c = corpus_by_profile[prof]
    n_tr = write_jsonl(c['train'], DATA_DIR / f'train_{prof}.jsonl')
    n_va = write_jsonl(c['validation'], DATA_DIR / f'val_{prof}.jsonl')
    rows = c['validation'][:500]
    lost = sum(1 for r in rows if not TONE.search(r['noisy'])) / len(rows) * 100
    print(f'{prof:<14}{n_tr:>9,}{n_va:>8,}{lost:>17.1f}%')

TRAIN_HARD, VAL_HARD = str(DATA_DIR / 'train_asr_hard.jsonl'), str(DATA_DIR / 'val_asr_hard.jsonl')
TRAIN_WHISPER, VAL_WHISPER = str(DATA_DIR / 'train_whisper_like.jsonl'), str(DATA_DIR / 'val_whisper_like.jsonl')
TRAIN_LEG, VAL_LEG = str(DATA_DIR / 'train_legacy.jsonl'), str(DATA_DIR / 'val_legacy.jsonl')

for name in ('train_asr_hard.jsonl', 'train_whisper_like.jsonl'):
    destination = DATA_DRIVE / name
    copyfile(DATA_DIR / name, destination)
    print(f'Đã lưu {destination} ({destination.stat().st_size:,} byte)', flush=True)


In [ ]:
# 4 — Train Lần 5 và lưu checkpoint trên Drive
import os
import subprocess
import sys
from src.models.train_seq2seq import resolve_output_dir

VARIANT = 'asr_hard'  # GD1: asr_hard; GD2: whisper_like; đối chứng: legacy
SMOKE = True          # False để train đầy đủ: 3 epoch, batch 16
LABEL_SMOOTHING = 0.1

DATA_OF = {
    'asr_hard': (TRAIN_HARD, VAL_HARD, 'asr_hard'),
    'whisper_like': (TRAIN_WHISPER, VAL_WHISPER, 'whisper'),
    'legacy': (TRAIN_LEG, VAL_LEG, 'legacy'),
}
TRAIN_F, VAL_F, TAG = DATA_OF[VARIANT]
mode = 'smoke10pct' if SMOKE else 'full'
RUN_TAG = f'lan5_{TAG}_{mode}_ls{int(LABEL_SMOOTHING * 10)}'
RUN_DIR_BASE = str(CKPT_ROOT / 'bmd1905-sub-corrector')
train_command = [sys.executable, '-u', '-m', 'src.models.train_seq2seq']

def run_with_live_output(command):
    env = {**os.environ, 'TQDM_MININTERVAL': '5'}
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, env=env) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
            exit_code = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            raise
    if exit_code:
        raise subprocess.CalledProcessError(exit_code, command)

DEST = None
run_with_live_output(train_command + ['--list_runs', '--output_dir', RUN_DIR_BASE])
run_dir = resolve_output_dir(RUN_DIR_BASE, RUN_TAG, overwrite=False)
print(f'Bài toán: {VARIANT} | Chế độ: {mode} | Lưu vào: {run_dir}', flush=True)
print(f'Dữ liệu: {TRAIN_F}', flush=True)
print('Đang khởi động script train và nạp thư viện…', flush=True)
run_with_live_output(train_command + [
    '--model_name', 'bmd1905/vietnamese-correction-v2',
    '--train_file', TRAIN_F, '--val_file', VAL_F, '--output_dir', str(run_dir),
    '--epochs', '1' if SMOKE else '3', '--batch_size', '8' if SMOKE else '16',
    '--lr', '3e-5', '--label_smoothing', str(LABEL_SMOOTHING),
] + (['--num_train_split', '10%'] if SMOKE else []))
DEST = run_dir


In [ ]:
# 5 — Benchmark Lần 5 với hai baseline trên Gold Test
import json
import subprocess
import sys
from pathlib import Path

BENCHMARK_CHECKPOINT = ""
if BENCHMARK_CHECKPOINT:
    DEST = Path(BENCHMARK_CHECKPOINT)

if not globals().get("DEST") or not (DEST / "config.json").is_file():
    raise FileNotFoundError("Chạy xong cell 4 hoặc điền BENCHMARK_CHECKPOINT hợp lệ.")

BENCHMARK_DIR = DEST / "benchmark"
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)
print(f"Kết quả sẽ lưu tại: {BENCHMARK_DIR}")

DEFAULT_DECODING = {
    "num_beams": 5,
    "repetition_penalty": 1.2,
    "length_penalty": 1.0,
    "no_repeat_ngram_size": 3,
}
decoding_path = DEST / "decoding.json"
DECODING = (
    json.loads(decoding_path.read_text(encoding="utf-8"))
    if decoding_path.is_file()
    else DEFAULT_DECODING
)

TEST_SUITES = {
    "sample25": "data/gold_test/gd1_basic/sample_test.jsonl",
    "unaccent": "data/gold_test/gd1_basic/basic_test_unaccent_all.jsonl",
    "asrhard": "data/gold_test/gd1_basic/basic_test_asr_hard.jsonl",
    "multisource": "data/gold_test/gd1_basic/basic_test_multisource.jsonl",
    "kichban": "data/gold_test/gd2_real_asr/kichban_test.jsonl",
}

missing = [p for p in TEST_SUITES.values() if not Path(p).is_file()]
if missing:
    raise FileNotFoundError(f"Thiếu tập test: {missing}")

decoding_args = [
    arg
    for key, value in DECODING.items()
    for arg in (f"--{key}", str(value))
]

for tag, test_file in TEST_SUITES.items():
    print(f"\n--- {DEST.name} / {tag} ---", flush=True)
    subprocess.run([
        sys.executable, "-m", "src.evaluation.run_benchmark",
        "--test_file", test_file,
        "--model_name_or_path", str(DEST),
        "--output_report", str(BENCHMARK_DIR / f"benchmark_{tag}.md"),
        "--output_json", str(BENCHMARK_DIR / f"metrics_{tag}.json"),
        "--output_log", str(BENCHMARK_DIR / f"benchmark_{tag}_log.txt"),
        "--output_excel", str(BENCHMARK_DIR / f"metrics_{tag}.xlsx"),
        "--batch_size", "16",
        *decoding_args,
    ], check=True)

In [ ]:
# 6 — Tùy chọn: benchmark Lần 5 trên corpus riêng: câu sạch và câu nhiễu
import subprocess
import sys
from tempfile import TemporaryDirectory

corpus_files = sorted(path for path in CORPUS_DRIVE.glob('*.txt') if path.is_file())
if corpus_files:
    print(f'Colab thấy {len(corpus_files)} tệp .txt trong {CORPUS_DRIVE}', flush=True)
    for path in corpus_files:
        print(f'  - {path.name}', flush=True)
    private_output = CORPUS_DRIVE / 'benchmark_runs' / DEST.name
    with TemporaryDirectory(prefix='corpus_eval_') as tmp:
        subprocess.run([sys.executable, '-m', 'src.data_processing.build_corpus_test',
                        '--corpus_dir', str(CORPUS_DRIVE), '--out_dir', tmp], check=True)
        corpus_suites = {
            'identity': Path(tmp) / 'corpus_identity.jsonl',
            'noisy': Path(tmp) / 'corpus_noisy.jsonl',
        }
        benchmark_suites(DEST, corpus_suites, private_output)
    print(f'Kết quả corpus trên Drive riêng: {private_output}')
else:
    print(f'Chưa thấy tệp .txt trong {CORPUS_DRIVE} — bỏ qua test corpus riêng.')


In [ ]:
# 7 — Tùy chọn: chọn cấu hình giải mã cho Lần 5 bằng validation
import json
import subprocess
import sys

# Chạy sau cell 4. Tách tiến trình để giải phóng VRAM trước khi benchmark lại.
tune_code = r'''
import json
import sys
from itertools import islice
from pathlib import Path
from src.models.inference import Seq2SeqCorrector
from src.evaluation.metrics import evaluate_predictions

model_dir, val_file = map(Path, sys.argv[1:])
with val_file.open(encoding='utf-8') as rows:
    dev = [json.loads(line) for line in islice(rows, 500)]
srcs = [r['noisy'] for r in dev]
refs = [r['clean'] for r in dev]
print(f'Grid-search trên {len(dev)} câu VAL\n')

GRID = [
    {'num_beams': 5, 'repetition_penalty': 1.20, 'length_penalty': 1.0, 'no_repeat_ngram_size': 3},
    {'num_beams': 5, 'repetition_penalty': 1.00, 'length_penalty': 1.0, 'no_repeat_ngram_size': 3},
    {'num_beams': 5, 'repetition_penalty': 1.05, 'length_penalty': 1.0, 'no_repeat_ngram_size': 0},
    {'num_beams': 8, 'repetition_penalty': 1.00, 'length_penalty': 1.0, 'no_repeat_ngram_size': 3},
    {'num_beams': 4, 'repetition_penalty': 1.00, 'length_penalty': 1.2, 'no_repeat_ngram_size': 3},
]

decoder = Seq2SeqCorrector(model_name_or_path=str(model_dir), use_fallback=False)

hdr = f'{"beams":>6}{"rep_pen":>9}{"len_pen":>9}{"ngram":>7}{"EM%":>8}{"BLEU":>8}{"WER%":>8}{"CER%":>8}{"Ed-F1%":>9}'
print(hdr)
print('-' * len(hdr))
scored = []
for cfg in GRID:
    decoder.set_decoding(**cfg)
    preds = decoder.predict_batch(srcs, batch_size=32)
    m = evaluate_predictions(srcs, preds, refs)
    scored.append((m['exact_match'], m['edit_f1'], cfg, m))
    print(f'{cfg["num_beams"]:>6}{cfg["repetition_penalty"]:>9.2f}{cfg["length_penalty"]:>9.1f}'
          f'{cfg["no_repeat_ngram_size"]:>7}{m["exact_match"]:>8.2f}{m["bleu"]:>8.2f}'
          f'{m["wer"]:>8.2f}{m["cer"]:>8.2f}{m["edit_f1"]:>9.2f}'
          + ('   <- mặc định' if cfg is GRID[0] else ''))

scored.sort(key=lambda t: (t[0], t[1]), reverse=True)
BEST_DECODING = scored[0][2]
(model_dir / 'decoding.json').write_text(json.dumps(BEST_DECODING, indent=2), encoding='utf-8')
print(f'CHỐT: {BEST_DECODING}. Chạy cell 5 để áp dụng khi benchmark.')
'''
manifest = json.loads((DEST / 'manifest.json').read_text(encoding='utf-8'))
val_file = manifest['dataset']['val_file']
if not val_file or not Path(val_file).is_file():
    raise FileNotFoundError('Cần sinh lại validation bằng cell 3 cho checkpoint này.')
subprocess.run([sys.executable, '-c', tune_code, str(DEST), val_file], check=True)


In [ ]:
# 8 — Tùy chọn: mở checkpoint tham chiếu Lần 4
import re
import shutil
import zipfile

REF = CKPT_ROOT / 'ref_lan4_legacy'
if (REF / 'config.json').is_file():
    print(f'Đã giải nén sẵn: {REF}')
else:
    zips = sorted(p for p in CKPT_ROOT.glob('*.zip') if 'bmd1905' in p.name.lower())
    if not zips:
        raise FileNotFoundError(f'Không thấy ZIP có bmd1905 trong {CKPT_ROOT}.')
    lan4 = [p for p in zips if re.search(r'(lan|train)[^0-9]*4', p.name, re.I)]
    REF_ZIP = (lan4 or zips)[0]
    print(f'Mốc so sánh: {REF_ZIP.name}')
    with zipfile.ZipFile(REF_ZIP) as archive:
        members = [n for n in archive.namelist()
                   if not n.endswith('/') and not any(p.startswith('checkpoint-') for p in n.split('/'))]
        anchor = next((n for n in members if n.split('/')[-1] == 'config.json'), None)
        if anchor is None:
            raise RuntimeError(f'{REF_ZIP.name} không có config.json của mô hình gốc.')
        root = anchor[:-len('config.json')]
        REF.mkdir(parents=True, exist_ok=True)
        for member in [n for n in members if n.startswith(root) and n != anchor] + [anchor]:
            target = REF / member[len(root):]
            if not target.resolve().is_relative_to(REF.resolve()):
                raise ValueError(f'Đường dẫn không hợp lệ trong ZIP: {member}')
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open('wb') as output:
                shutil.copyfileobj(source, output)
    print(f'Đã giải nén vào {REF}')


In [ ]:
# 9 — Tùy chọn: benchmark Lần 4 với thiết lập và tập test của Lần 5
benchmark_suites(REF, TEST_SUITES, REF)


In [ ]:
# 10 — Tùy chọn: ghi manifest của Lần 4 từ log Colab gốc
import json
from datetime import datetime

(REF / 'manifest.json').write_text(json.dumps({
    'run_name': REF.name,
    'created_at': datetime.now().astimezone().isoformat(timespec='seconds'),
    'ghi_chu': 'ĐIỀN THỦ CÔNG từ log Lần 4, không do script tự sinh',
    'base_model': 'bmd1905/vietnamese-correction-v2',
    'device': 'cuda',
    'gpu': 'Tesla T4',
    'hyperparams': {'num_train_epochs': 3, 'per_device_train_batch_size': 16,
                    'learning_rate': 3e-05, 'max_length': 128, 'label_smoothing': 0.0,
                    'padding': 'static', 'fp16': True},
    'dataset': {'train_size': 90000, 'val_size': 10000},
    'results': {'train_loss_final': 0.03736,
                'eval_loss_per_epoch': [0.02764, 0.01632, 0.01307],
                'best_eval_loss': 0.01307,
                'train_runtime_sec': 7941.0,
                'total_steps': 16875},
}, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Đã ghi {REF / "manifest.json"}')


In [ ]:
# 11 — Tùy chọn: so sánh các checkpoint trên cùng một tập test
from src.evaluation.compare_runs import best_values, collect_runs, render_markdown, sort_rows

COMPARE_ABLATION = False  # True sau cell 13 để so Lần 5 với B/C
COMPARE_TAG = 'multisource'
run_dirs = [DEST, DEST_B, DEST_C] if COMPARE_ABLATION else [DEST, REF]
rows, settings = [], []
for directory in run_dirs:
    row = next((r for r in collect_runs(directory.parent) if r['path'] == directory), None)
    if row is None:
        raise FileNotFoundError(f'Thiếu manifest.json trong {directory}')
    bench_path = directory / f'metrics_{COMPARE_TAG}.json'
    bench = json.loads(bench_path.read_text(encoding='utf-8'))
    if bench.get('engine') != 'seq2seq':
        raise ValueError(f'{bench_path} không phải kết quả Seq2Seq.')
    row.update(engine=bench['engine'], decoding=bench['decoding'], n_test=bench['num_samples'],
               metrics=next(m for name, m in bench['results'].items() if name.startswith('Seq2Seq')))
    settings.append((bench['test_file'], bench['num_samples'], bench['decoding']))
    rows.append(row)
if any(setting != settings[0] for setting in settings[1:]):
    raise ValueError('Các checkpoint cần được chấm cùng tập test và cùng cấu hình giải mã.')

rows = sort_rows(rows, 'exact_match')
comparison = 'ablation' if COMPARE_ABLATION else 'lan4'
out = DEST / f'so_sanh_{comparison}_{COMPARE_TAG}.md'
out.write_text(render_markdown(rows, best_values(rows)), encoding='utf-8')
print(f'Đã lưu bảng tại {out}')


In [ ]:
# 12 — Tùy chọn: train B/C để đo riêng tác động của label smoothing và hồ sơ nhiễu
# Bỏ qua nếu Colab sắp hết phiên.
from src.models.train_seq2seq import resolve_output_dir

# (B) asr_hard KHÔNG label smoothing -> cô lập tác động của LS
DEST_B = resolve_output_dir(str(CKPT_ROOT / 'bmd1905-sub-corrector'), 'lan5_asr_hard_ls00', overwrite=False)
!python -m src.models.train_seq2seq \
    --model_name 'bmd1905/vietnamese-correction-v2' \
    --train_file '{TRAIN_HARD}' --val_file '{VAL_HARD}' \
    --output_dir '{DEST_B}' \
    --epochs 3 --batch_size 16 --lr 3e-5 --label_smoothing 0.0

# (C) nhiễu LEGACY (như Lần 4) nhưng CÓ pad động + LS -> cô lập tác động của NHIỄU
DEST_C = resolve_output_dir(str(CKPT_ROOT / 'bmd1905-sub-corrector'), 'lan5_legacy_ls01', overwrite=False)
!python -m src.models.train_seq2seq \
    --model_name 'bmd1905/vietnamese-correction-v2' \
    --train_file '{TRAIN_LEG}' --val_file '{VAL_LEG}' \
    --output_dir '{DEST_C}' \
    --epochs 3 --batch_size 16 --lr 3e-5 --label_smoothing 0.1


In [ ]:
# 13 — Tùy chọn: benchmark B/C rồi chạy cell 11 với COMPARE_ABLATION = True
for run in (DEST_B, DEST_C):
    benchmark_suites(run, {'multisource': TEST_SUITES['multisource']}, run)
